In [1]:
import numpy as np
import json
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

d:\ML_project\Tomato_Disease_Detection\venv\lib\site-packages\google\api_core\_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.0). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
d:\ML_project\Tomato_Disease_Detection\venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
d:\ML_project\Tomato_Disease_Detection\venv\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will upda

In [2]:
model = load_model("../trained_model/new_tomato_disease_model.h5")

In [3]:
with open("class_indices.json", "r") as f:
    class_indices = json.load(f)

In [4]:
idx_to_class = {v: k for k, v in class_indices.items()}

In [7]:
def predict_disease(img_path, threshold=0.65):
    # Load & preprocess image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(img_array, verbose=0)[0]
    top_index = np.argmax(predictions)
    top_class = idx_to_class[top_index]
    confidence = float(predictions[top_index])

    # Check threshold
    if confidence < threshold:
        top_class = "Uncertain: Retake Image"

    # Top 3 predictions (for reporting)
    top3_indices = predictions.argsort()[-3:][::-1]
    top3 = [(idx_to_class[i], float(predictions[i])) for i in top3_indices]

    return top_class, confidence, top3


In [9]:
# ----------------- Early Blight Mini-Test -----------------
import os
import random
test_dir = "../Data_sets_split/test/Tomato___Early_blight"
images = random.sample(os.listdir(test_dir), 20)

correct = 0
wrong_confidences = []
correct_confidences = []
uncertain_count = 0

for img_name in images:
    path = os.path.join(test_dir, img_name)
    pred_class, conf, top3 = predict_disease(path, threshold=0.65)

    if pred_class == "Tomato___Early_blight":
        correct += 1
        correct_confidences.append(conf)
    elif pred_class.startswith("Uncertain"):
        uncertain_count += 1
        wrong_confidences.append(conf)
    else:
        wrong_confidences.append(conf)

    print(f"Image: {img_name}")
    print(f"Predicted: {pred_class}")
    print(f"Confidence: {conf:.4f}")
    print(f"Top3: {top3}")
    print("------")

# ----------------- Results Summary -----------------
total_tested = len(images)
print("\nRESULTS SUMMARY")
print(f"Total tested: {total_tested}")
print(f"Correct predictions: {correct}")
print(f"Uncertain predictions: {uncertain_count}")
print(f"Wrong predictions: {total_tested - correct - uncertain_count}")

if correct_confidences:
    print("Avg confidence (correct):", sum(correct_confidences)/len(correct_confidences))
if wrong_confidences:
    print("Avg confidence (wrong/uncertain):", sum(wrong_confidences)/len(wrong_confidences))

Image: 1bff3ab2-9e99-4d4b-ad0e-acd943324050___RS_Erly.B 6460.JPG
Predicted: Uncertain: Retake Image
Confidence: 0.5586
Top3: [('Tomato___Early_blight', 0.5586044192314148), ('Tomato___Leaf_Mold', 0.20003600418567657), ('Tomato___Bacterial_spot', 0.14672105014324188)]
------
Image: fb3352cc-9189-4214-8b94-a60c6a392c96___RS_Erly.B 9600.JPG
Predicted: Tomato___Early_blight
Confidence: 0.7886
Top3: [('Tomato___Early_blight', 0.7885729670524597), ('Tomato___Late_blight', 0.08116888999938965), ('Tomato___Bacterial_spot', 0.06828897446393967)]
------
Image: 64fd8cac-b985-45c1-9a5d-5f2f49381047___RS_Erly.B 9494.JPG
Predicted: Tomato___Early_blight
Confidence: 0.8441
Top3: [('Tomato___Early_blight', 0.8440779447555542), ('Tomato___Septoria_leaf_spot', 0.11376972496509552), ('Tomato___Late_blight', 0.014687344431877136)]
------
Image: a6d7438a-35f6-40fb-ab5c-a46cdfef7e83___RS_Erly.B 9594.JPG
Predicted: Tomato___Early_blight
Confidence: 0.9078
Top3: [('Tomato___Early_blight', 0.9078028798103333),